# Deep Learning-Based Cardiovascular Disease Risk Prediction Using Retinal Fundus Images

**Domain:** Deep Learning · Medical Image Processing · Computer Vision · Healthcare

**Dataset:** ODIR-5K (Ocular Disease Intelligent Recognition Dataset)
Source: https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k

---

## ⚠️ Important Methodological Note (read before using this notebook)

The ODIR-5K dataset does **not** contain direct cardiovascular disease (CVD) outcome labels
(e.g. confirmed heart attack, stroke, angiography results, Framingham score, etc.). It contains
**ocular disease labels**: Normal (N), Diabetes (D), Glaucoma (G), Cataract (C), AMD (A),
Hypertension (H), Pathological Myopia (M), Other (O).

This notebook therefore does two things, and keeps them clearly separated:

1. **Primary, dataset-faithful task** — train a CNN (EfficientNetB0 transfer learning) to classify
   the actual ODIR disease labels from retinal fundus images. This part is fully supported by the
   data and is medically defensible.
2. **Secondary, heuristic task** — derive a 3-class **proxy** cardiovascular risk label
   (`Low` / `Medium` / `High`) from the ODIR disease labels, based on the *known association*
   between retinal vascular damage (driven by diabetes/hypertension) and systemic cardiovascular
   risk, and train the same architecture on that proxy label.

**The proxy risk label is a heuristic for this educational/research project — it has not been
clinically validated and must not be presented as a diagnostic CVD risk score.** If you have
access to a dataset with real CVD outcomes (e.g. UK Biobank retinal cohort, EyePACS with linked
cardiovascular events), swap the label-generation step in Section 4 for the real labels and the
rest of the pipeline (preprocessing → model → training → evaluation) stays the same.

---

## Pipeline Overview

```
Retinal Fundus Image
        ↓
Image Preprocessing (resize 224x224, normalize, augment)
        ↓
EfficientNetB0 (ImageNet transfer learning)
        ↓
Feature Extraction → Classification Head
        ↓
Risk Score Generation → Low / Medium / High
```


## 1. Setup & Imports

In [ ]:
# If running in a fresh environment (e.g. Colab), uncomment to install dependencies:
# !pip install tensorflow opencv-python-headless pandas numpy matplotlib scikit-learn openpyxl pillow seaborn

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    # ---- Paths: adjust to where you extracted the Kaggle ODIR-5K archive ----
    "ARCHIVE_DIR": "archive",
    "FULL_DF_CSV": "archive/full_df.csv",
    "IMAGES_DIR": "archive/preprocessed_images",   # contains e.g. 0_left.jpg, 0_right.jpg
    "MODEL_DIR": "models",

    # ---- Image / training params ----
    "IMG_SIZE": (224, 224),
    "BATCH_SIZE": 32,
    "EPOCHS_FROZEN": 15,
    "EPOCHS_FINETUNE": 15,
    "LEARNING_RATE": 1e-4,
    "FINE_TUNE_LR": 1e-5,
    "FINE_TUNE_AT_LAYER": -30,   # unfreeze last 30 layers of EfficientNetB0 during fine-tuning
    "VAL_SPLIT": 0.10,
    "TEST_SPLIT": 0.10,
}

RISK_CLASSES = ["Low", "Medium", "High"]
NUM_CLASSES = len(RISK_CLASSES)

os.makedirs(CONFIG["MODEL_DIR"], exist_ok=True)
CONFIG


## 3. Load & Explore Metadata

`full_df.csv` (the flattened version of the ODIR metadata that ships with the Kaggle
"preprocessed_images" copy of the dataset) has one row per **patient**, with columns for both
eyes and the multi-label disease flags `N, D, G, C, A, H, M, O`.


In [ ]:
df = pd.read_csv(CONFIG["FULL_DF_CSV"])
print("Shape:", df.shape)
df.head()


In [ ]:
label_cols = ["N", "D", "G", "C", "A", "H", "M", "O"]

# Sanity-check that the expected label columns exist
missing = [c for c in label_cols if c not in df.columns]
if missing:
    raise ValueError(f"Expected ODIR label columns missing from full_df.csv: {missing}")

print("Patients:", len(df))
print("\nLabel prevalence (a patient can have more than one label):")
print(df[label_cols].sum().sort_values(ascending=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df[label_cols].sum().sort_values(ascending=False).plot(kind="bar", ax=ax, color="#3b6fa0")
ax.set_title("ODIR-5K Disease Label Frequency")
ax.set_ylabel("Number of patients")
plt.tight_layout()
plt.show()


## 4. Building Two Label Sets

We expand each **patient** row into two **eye-level samples** (left + right), since the CNN
classifies individual fundus images. Both eyes inherit the patient-level disease flags (a common,
documented simplification for this dataset since per-eye diagnostic keywords are noisier than the
aggregated flags).

For each eye-level sample we generate:

- `disease_label` — the dominant ODIR disease flag (dataset-faithful, Section 1 task).
- `risk_label` — the heuristic 3-class CVD-proxy risk (Section 2 task), via `map_to_cvd_risk()`.

### Heuristic mapping used (documented, not clinical ground truth)

| Condition present | Rationale | Proxy risk |
|---|---|---|
| Hypertension (H) or Diabetes (D) | Directly damages retinal microvasculature; strongest literature link to systemic CVD risk | **High** |
| Glaucoma (G), AMD (A), Pathological Myopia (M), Other (O) — and no H/D | Associated with vascular/structural changes but weaker/less direct CVD evidence | **Medium** |
| Normal (N) only, no other flag | No detected retinal abnormality | **Low** |
| Multiple non-H/D flags together | Compounding abnormality | **Medium** |

Adjust this table freely — it is a placeholder until real outcome labels are available.


In [ ]:
def get_dominant_disease(row, cols):
    """Pick a single dataset-faithful disease label per sample for the classification task.
    Priority order favors clinically 'stronger' single findings when multiple flags are set."""
    priority = ["H", "D", "G", "A", "M", "C", "O", "N"]
    for p in priority:
        if row[p] == 1:
            return p
    return "O"

def map_to_cvd_risk(row):
    """Heuristic, NOT clinically validated. See markdown above for rationale."""
    if row["H"] == 1 or row["D"] == 1:
        return "High"
    if row["G"] == 1 or row["A"] == 1 or row["M"] == 1 or row["O"] == 1:
        return "Medium"
    if row["N"] == 1:
        return "Low"
    return "Medium"  # fallback for any unmapped edge case

def build_eye_level_dataframe(df, images_dir, label_cols):
    records = []
    for _, row in df.iterrows():
        disease = get_dominant_disease(row, label_cols)
        risk = map_to_cvd_risk(row)
        for side, fname_col in [("left", "Left-Fundus"), ("right", "Right-Fundus")]:
            if fname_col not in row or pd.isna(row[fname_col]):
                continue
            img_path = os.path.join(images_dir, row[fname_col])
            records.append({
                "patient_id": row.get("ID", None),
                "age": row.get("Patient Age", None),
                "sex": row.get("Patient Sex", None),
                "eye": side,
                "image_path": img_path,
                "disease_label": disease,
                "risk_label": risk,
            })
    return pd.DataFrame(records)

eye_df = build_eye_level_dataframe(df, CONFIG["IMAGES_DIR"], label_cols)

# Keep only rows whose image file actually exists on disk
eye_df["exists"] = eye_df["image_path"].apply(os.path.exists)
print("Eye-level samples before filtering:", len(eye_df))
eye_df = eye_df[eye_df["exists"]].drop(columns="exists").reset_index(drop=True)
print("Eye-level samples with image found on disk:", len(eye_df))

eye_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eye_df["disease_label"].value_counts().plot(kind="bar", ax=axes[0], color="#3b6fa0")
axes[0].set_title("Dataset-faithful disease label distribution")

eye_df["risk_label"].value_counts().reindex(RISK_CLASSES).plot(kind="bar", ax=axes[1], color="#a04b3b")
axes[1].set_title("Heuristic CVD-proxy risk distribution")
plt.tight_layout()
plt.show()


## 5. Train / Validation / Test Split

This notebook trains on `risk_label` (the proxy CVD-risk task) end-to-end. The exact same code
works for `disease_label` — just change `TARGET_COL` below.


In [ ]:
TARGET_COL = "risk_label"   # change to "disease_label" to run the dataset-faithful task instead
CLASS_NAMES = sorted(eye_df[TARGET_COL].unique()) if TARGET_COL != "risk_label" else RISK_CLASSES
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
eye_df["label_idx"] = eye_df[TARGET_COL].map(class_to_idx)

train_df, temp_df = train_test_split(
    eye_df, test_size=(CONFIG["VAL_SPLIT"] + CONFIG["TEST_SPLIT"]),
    stratify=eye_df["label_idx"], random_state=SEED
)
val_frac_of_temp = CONFIG["VAL_SPLIT"] / (CONFIG["VAL_SPLIT"] + CONFIG["TEST_SPLIT"])
val_df, test_df = train_test_split(
    temp_df, test_size=(1 - val_frac_of_temp),
    stratify=temp_df["label_idx"], random_state=SEED
)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
print("\nClass balance (train):")
print(train_df[TARGET_COL].value_counts(normalize=True).round(3))


## 6. Image Loading, Preprocessing & Augmentation (tf.data pipeline)

In [ ]:
IMG_SIZE = CONFIG["IMG_SIZE"]
BATCH_SIZE = CONFIG["BATCH_SIZE"]

def load_and_preprocess(path, label, img_size=IMG_SIZE):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0       # scale to [0, 1]
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return image, label

def make_dataset(dataframe, shuffle=False, batch_size=BATCH_SIZE):
    paths = dataframe["image_path"].values
    labels = dataframe["label_idx"].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df,   shuffle=False)
test_ds  = make_dataset(test_df,  shuffle=False)

# On-the-fly augmentation, applied only inside the model during training (see build_model)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")


In [ ]:
# Quick visual sanity check of a training batch
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].numpy())
    cls = CLASS_NAMES[np.argmax(labels[i].numpy())]
    ax.set_title(cls)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 7. Model Architecture — EfficientNetB0 Transfer Learning

ImageNet-pretrained `EfficientNetB0` as the backbone, with augmentation baked into the model graph
(active only when `training=True`), a pooling + dropout + dense classification head, and a final
softmax over the risk classes.


In [ ]:
def build_model(input_shape=(224, 224, 3), num_classes=NUM_CLASSES, base_trainable=False):
    base_model = EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=input_shape
    )
    base_model.trainable = base_trainable

    inputs = tf.keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = base_model(x, training=base_trainable)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="cvd_risk_efficientnetb0")
    return model, base_model

model, base_model = build_model(input_shape=IMG_SIZE + (3,), base_trainable=False)
model.compile(
    optimizer=optimizers.Adam(learning_rate=CONFIG["LEARNING_RATE"]),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


## 8. Callbacks

In [ ]:
checkpoint_path = os.path.join(CONFIG["MODEL_DIR"], "cvd_model_best.keras")

callback_list = [
    callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
]


## 9. Phase 1 — Train Classification Head (Backbone Frozen)

In [ ]:
history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["EPOCHS_FROZEN"],
    callbacks=callback_list
)


## 10. Phase 2 — Fine-Tune Top Layers of the Backbone

Unfreeze the last block(s) of EfficientNetB0 and continue training with a much smaller learning
rate, so the pretrained ImageNet features adapt gently to retinal images instead of being
overwritten.


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:CONFIG["FINE_TUNE_AT_LAYER"]]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=CONFIG["FINE_TUNE_LR"]),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Fine-tuning {trainable_count} of {len(base_model.layers)} backbone layers")

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["EPOCHS_FINETUNE"],
    callbacks=callback_list
)


## 11. Training Curves

In [ ]:
def combine_histories(h1, h2):
    combined = {}
    for k in h1.history:
        combined[k] = h1.history[k] + h2.history.get(k, [])
    return combined

hist = combine_histories(history_frozen, history_finetune)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(hist["accuracy"], label="Train")
axes[0].plot(hist["val_accuracy"], label="Validation")
axes[0].axvline(CONFIG["EPOCHS_FROZEN"], color="gray", linestyle="--", label="Fine-tune start")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist["loss"], label="Train")
axes[1].plot(hist["val_loss"], label="Validation")
axes[1].axvline(CONFIG["EPOCHS_FROZEN"], color="gray", linestyle="--", label="Fine-tune start")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 12. Evaluation on the Held-Out Test Set

Accuracy, precision, recall, F1, confusion matrix, classification report, and per-class ROC/AUC.


In [ ]:
y_true, y_pred_probs = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — CVD Risk Proxy")
plt.tight_layout()
plt.show()


In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

plt.figure(figsize=(6, 5))
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Per-Class ROC Curves — CVD Risk Proxy")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Save the Trained Model

In [ ]:
final_model_path = os.path.join(CONFIG["MODEL_DIR"], "cvd_model.keras")
model.save(final_model_path)
print("Saved model to:", final_model_path)


## 14. Grad-CAM — Visualizing What the Model Looks At

Grad-CAM highlights the retinal regions (typically vessels and optic-disc area) that most
influenced a given prediction. This is essential for any clinically-adjacent tool: it lets a
clinician sanity-check that the model is keying off vascular structure rather than spurious
artifacts (image borders, camera vignetting, text overlays, etc.).


In [ ]:
def get_last_conv_layer_name(keras_model):
    for layer in reversed(keras_model.layers):
        if len(layer.output_shape) == 4:   # has spatial dims (batch, H, W, C)
            return layer.name
    raise ValueError("No 4D conv-like layer found")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(image_rgb_float, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (image_rgb_float.shape[1], image_rgb_float.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    base_img = image_rgb_float
    overlaid = heatmap_color * alpha + base_img * (1 - alpha)
    return np.clip(overlaid, 0, 1)

# Find the last 4D (spatial) layer inside the EfficientNetB0 backbone for Grad-CAM
last_conv_name = get_last_conv_layer_name(base_model)
print("Using layer for Grad-CAM:", last_conv_name)

# Build a sub-model so Grad-CAM can be computed wrt the backbone's own output tensor
gradcam_model = tf.keras.Model(model.inputs, [base_model.get_layer(last_conv_name).output, model.output])

sample_images, sample_labels = next(iter(test_ds))
sample_img = sample_images[0:1]

heatmap = make_gradcam_heatmap(sample_img, model, last_conv_layer_name=last_conv_name)
overlay = overlay_gradcam(sample_img[0].numpy(), heatmap)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(sample_img[0].numpy()); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM Heatmap"); axes[1].axis("off")
axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
plt.tight_layout()
plt.show()


## 15. Inference Function for a New Fundus Image

In [ ]:
def predict_risk(image_path, model=model, class_names=CLASS_NAMES, img_size=IMG_SIZE, show=True):
    """Run the trained model on a single fundus image and return (predicted_class, probabilities)."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0
    batch = tf.expand_dims(image, axis=0)

    probs = model.predict(batch, verbose=0)[0]
    pred_class = class_names[int(np.argmax(probs))]

    if show:
        plt.figure(figsize=(4, 4))
        plt.imshow(image.numpy())
        plt.title(f"Predicted: {pred_class} ({probs.max():.1%} confidence)")
        plt.axis("off")
        plt.show()
        for c, p in zip(class_names, probs):
            print(f"  {c:8s}: {p:.3f}")

    return pred_class, probs

# Example usage (uncomment and point to a real file):
# predict_risk(test_df.iloc[0]["image_path"])


## 16. Limitations & Future Scope

- **Label quality**: the `risk_label` used here is a heuristic proxy derived from ocular disease
  flags, not a validated cardiovascular outcome. Treat all "risk" numbers from this notebook as
  illustrative, not diagnostic.
- **Per-eye independence assumption**: left/right eyes from the same patient are treated as
  independent samples for training; in a rigorous setup, split by **patient ID** (not by image)
  so a patient's two eyes never appear in both train and test — this notebook's `train_test_split`
  should be adapted to split on `patient_id` for a stricter evaluation.
- **Multimodal extension**: combine retinal images with patient age, sex, blood pressure,
  cholesterol, BMI, and smoking status (tabular branch + CNN branch fused before the final dense
  layers) once such fields are available.
- **Real CVD outcomes**: retrain the identical pipeline on a dataset with linked cardiovascular
  events (e.g. 10-year CVD incidence) for a clinically meaningful risk score.
- **Deployment**: wrap `predict_risk()` in a Flask/FastAPI endpoint or a Streamlit app for an
  interactive screening demo.
- **Alternative backbones**: `ResNet50`, `DenseNet121`, `MobileNetV2`, and `InceptionV3` can be
  swapped in by replacing `EfficientNetB0(...)` in `build_model()` — input preprocessing should
  match each architecture's expected scaling if you switch away from EfficientNet.
